# https://www.kaggle.com/datasets/lovishbansal123/adult-census-income

In [1]:
# Import necessary libraries

# pandas is used for data manipulation and analysis, especially for handling data in DataFrame format
import pandas as pd

# numpy is used for numerical operations, such as arrays, mathematical functions, etc.
import numpy as np

# accuracy_score from sklearn is used to calculate the accuracy of a model's predictions
from sklearn.metrics import accuracy_score

# CatBoostClassifier is the CatBoost library's implementation of a gradient boosting model for classification tasks
from catboost import CatBoostClassifier

# train_test_split from sklearn is used to split the dataset into training and testing subsets
from sklearn.model_selection import train_test_split

In [2]:
# Installing the model - CatBoost

# !pip install catboost

In [3]:
df = pd.read_csv('C:/Users/Admin/Desktop/ML/LogisticRegression/Adult Census ML/adult.csv')
df.head(5)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,?,77053,HS-grad,9,Widowed,?,Not-in-family,White,Female,0,4356,40,United-States,<=50K
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K
2,66,?,186061,Some-college,10,Widowed,?,Unmarried,Black,Female,0,4356,40,United-States,<=50K
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K


In [4]:
df['capital.gain'].value_counts()

# Too many outliers in capital.gain column

capital.gain
0        29849
15024      347
7688       284
7298       246
99999      159
         ...  
1111         1
4931         1
7978         1
5060         1
2538         1
Name: count, Length: 119, dtype: int64

In [5]:
# df['capital.gain_log'] = np.log1p(df['capital.gain'])

# No worries  - CatBoost model will handle it auto

In [6]:
# mode_occupation = df['occupation'].mode()[0]  # Get the most common occupation
# df['occupation'].replace('?', mode_occupation, inplace=True)

# mode_workclass = df['workclass'].mode()[0]  # Get the most common occupation
# df['workclass'].replace('?', mode_workclass, inplace=True)

# No worries  - CatBoost model will handle it auto

In [7]:
# label_encoder = LabelEncoder()
# hot_encoder = OneHotEncoder()

# No worries  - CatBoost model will handle it auto

In [8]:
# df = pd.get_dummies(df, columns=['workclass'], drop_first=True)

# No worries  - CatBoost model will handle it auto

In [9]:
df.info()

# Checking cols dtypes and their non-null count

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [10]:
df.isnull().sum()

# Checking for null values - but missing values were described as '?'

age               0
workclass         0
fnlwgt            0
education         0
education.num     0
marital.status    0
occupation        0
relationship      0
race              0
sex               0
capital.gain      0
capital.loss      0
hours.per.week    0
native.country    0
income            0
dtype: int64

In [11]:
X = df.drop(columns=['income'], axis=1)
y = df['income']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=0.2)

In [13]:
cat_features = ['workclass', 'education', 'marital.status', 'occupation', 
                'relationship', 'race', 'sex', 'native.country']

for col in cat_features:
    X_train[col] = X_train[col].replace('?', 'Missing').fillna('Missing').astype(str)
    X_test[col] = X_test[col].replace('?', 'Missing').fillna('Missing').astype(str)
    
# The catboost requires that the col in which categorizing is happening must be listed separately in the array
# When you replace '?' or NaN with a placeholder like 'Missing', CatBoost will interpret that as a distinct category 
# rather than an actual missing value. This means that CatBoost will treat 'Missing' as an additional category.
# However, CatBoost needs consistency in the way it handles categorical columns. The categorical columns should 
# be uniform in terms of data types (e.g., they should all be either strings or integers).

In [14]:
model = CatBoostClassifier(verbose=100)
model.fit(X_train, y_train, cat_features=cat_features)

Learning rate set to 0.041445
0:	learn: 0.6478124	total: 388ms	remaining: 6m 28s
100:	learn: 0.2916884	total: 8.07s	remaining: 1m 11s
200:	learn: 0.2785805	total: 17.3s	remaining: 1m 8s
300:	learn: 0.2704311	total: 25.6s	remaining: 59.4s
400:	learn: 0.2636546	total: 35.4s	remaining: 52.9s
500:	learn: 0.2596642	total: 44.4s	remaining: 44.2s
600:	learn: 0.2556525	total: 55.1s	remaining: 36.6s
700:	learn: 0.2520648	total: 1m 5s	remaining: 28.1s
800:	learn: 0.2484052	total: 1m 15s	remaining: 18.8s
900:	learn: 0.2453667	total: 1m 24s	remaining: 9.29s
999:	learn: 0.2423037	total: 1m 33s	remaining: 0us


In [16]:
# Predictions on training data
train_preds = model.predict(X_train)
train_accuracy = accuracy_score(y_train, train_preds)
print(f"Training Accuracy: {train_accuracy:.4f}")

# Predictions on test data
test_preds = model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_preds)
print(f"Test Accuracy: {test_accuracy:.4f}")

Training Accuracy: 0.8886
Test Accuracy: 0.8761
